# 1. INSPECT THE DATASET

In [2]:
import pandas as pd
import numpy as np

In [ ]:
DATA_PATH = "C:\\Users\\sayum\\Desktop\\smartenergy-ai\\data\\raw\\household_power_consumption.txt"

df = pd.read_csv(
    DATA_PATH,
    sep=";",          # Use semicolon as the separator for the dataset
    low_memory=False  # Use low_memory=False to avoid dtype inference issues with large files
)
print("Shape:", df.shape)

Shape: (2075259, 9)


# 2. MISSING VALUES

In [ ]:
missing_values = df.isnull().sum() #creates a boolean DataFrame where True indicates a missing/NaN value 
                                   #.sum() -- adds up all the True values (which count as 1) for each column

missing_percentage = (
    df.isnull().mean() * 100       # calculates the proportion (0-1) of missing values per column
).sort_values(ascending=False)     # sorts from highest percentage to lowest

missing_report = pd.DataFrame({    # Combines both Series into a single DataFrame
    "missing_count": missing_values, # First column shows count of missing values
    "missing_percentage": missing_percentage # Second column shows percentage of missing values
})

display(missing_report)

,missing_count,missing_percentage
Date,0,0.000000
Global_active_power,0,0.000000
Global_intensity,0,0.000000
Global_reactive_power,0,0.000000
Sub_metering_1,0,0.000000
Sub_metering_2,0,0.000000
Sub_metering_3,25979,1.251844
Time,0,0.000000
Voltage,0,0.000000


# 3. DUPLICATE RECORDS    

In [ ]:
duplicate_count = df.duplicated().sum()  #Check every row and tell whether it is a duplicate of a previous row.

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


# 4.DATA TYPES

In [7]:
print(df.dtypes)

Date                      object
Time                      object
Global_active_power       object
Global_reactive_power     object
Voltage                   object
Global_intensity          object
Sub_metering_1            object
Sub_metering_2            object
Sub_metering_3           float64
dtype: object


In [ ]:
# Combine Date and Time into one timestamp column
df["timestamp"] = pd.to_datetime(    # creating a new column called timestamp
    df["Date"] + " " + df["Time"],   # Select the Date column from the DataFrame AND concatenate it with the Time column, separated by a space
    errors="coerce"  # If there are any errors in parsing the date and time, they will be set to NaT (Not a Time) instead of raising an error
)
print(df[["Date", "Time", "timestamp"]].head()) # Check the result

C:\Users\sayum\AppData\Local\Temp\ipykernel_19548\1778988095.py:2: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["timestamp"] = pd.to_datetime(


         Date      Time           timestamp
0  16/12/2006  17:24:00 2006-12-16 17:24:00
1  16/12/2006  17:25:00 2006-12-16 17:25:00
2  16/12/2006  17:26:00 2006-12-16 17:26:00
3  16/12/2006  17:27:00 2006-12-16 17:27:00
4  16/12/2006  17:28:00 2006-12-16 17:28:00


In [ ]:
print("Invalid timestamps:", df["timestamp"].isna().sum())  #Count how many missing/invalid timestamps exist.

Invalid timestamps: 0


In [11]:
print(df.dtypes)

Date                             object
Time                             object
Global_active_power              object
Global_reactive_power            object
Voltage                          object
Global_intensity                 object
Sub_metering_1                   object
Sub_metering_2                   object
Sub_metering_3                  float64
timestamp                datetime64[ns]
dtype: object


# 5. CHECK INVALID VALUES

In [ ]:
print(df.describe())   # normally summarizes numeric columns. Note that Sub_metering_3 is treated as a numeric column at the moment.

       Sub_metering_3                      timestamp
count    2.049280e+06                        2075259
mean     6.458447e+00  2008-12-06 07:12:59.999994112
min      0.000000e+00            2006-12-16 17:24:00
25%      0.000000e+00            2007-12-12 00:18:30
50%      1.000000e+00            2008-12-06 07:13:00
75%      1.700000e+01            2009-12-01 14:07:30
max      3.100000e+01            2010-11-26 21:02:00
std      8.437154e+00                            NaN


# 6. Convert electricity columns to numeric

In [14]:
numeric_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

In [15]:
print(df.dtypes)

Date                             object
Time                             object
Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
timestamp                datetime64[ns]
dtype: object


In [16]:
print(df.isnull().sum())   # Check missing values AGAIN

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
timestamp                    0
dtype: int64


In [17]:
print(df[numeric_columns].describe())

       Global_active_power  Global_reactive_power       Voltage  \
count         2.049280e+06           2.049280e+06  2.049280e+06   
mean          1.091615e+00           1.237145e-01  2.408399e+02   
std           1.057294e+00           1.127220e-01  3.239987e+00   
min           7.600000e-02           0.000000e+00  2.232000e+02   
25%           3.080000e-01           4.800000e-02  2.389900e+02   
50%           6.020000e-01           1.000000e-01  2.410100e+02   
75%           1.528000e+00           1.940000e-01  2.428900e+02   
max           1.112200e+01           1.390000e+00  2.541500e+02   

       Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
count      2.049280e+06    2.049280e+06    2.049280e+06    2.049280e+06  
mean       4.627759e+00    1.121923e+00    1.298520e+00    6.458447e+00  
std        4.444396e+00    6.153031e+00    5.822026e+00    8.437154e+00  
min        2.000000e-01    0.000000e+00    0.000000e+00    0.000000e+00  
25%        1.400000e+00   

In [25]:
missing_rows = df[df["Global_active_power"].isna()]

print("Number of missing rows:", len(missing_rows))

display(
    missing_rows[
        ["Date", "Time", "timestamp"]
    ].head(20)
)

Number of missing rows: 25979


,Date,Time,timestamp
6839,21/12/2006,11:23:00,2006-12-21 11:23:00
6840,21/12/2006,11:24:00,2006-12-21 11:24:00
19724,30/12/2006,10:08:00,2006-12-30 10:08:00
19725,30/12/2006,10:09:00,2006-12-30 10:09:00
41832,14/1/2007,18:36:00,2007-01-14 18:36:00
61909,28/1/2007,17:13:00,2007-01-28 17:13:00
98254,22/2/2007,22:58:00,2007-02-22 22:58:00
98255,22/2/2007,22:59:00,2007-02-22 22:59:00
142588,25/3/2007,17:52:00,2007-03-25 17:52:00
190497,28/4/2007,00:21:00,2007-04-28 00:21:00


In [26]:
print(
    missing_rows[
        ["timestamp"]
    ].min()
)

print(
    missing_rows[
        ["timestamp"]
    ].max()
)

timestamp   2006-12-21 11:23:00
dtype: datetime64[ns]
timestamp   2010-10-24 15:35:00
dtype: datetime64[ns]


In [27]:
print(
    missing_rows["timestamp"].diff().value_counts().head(10)
)

timestamp
0 days 00:01:00     25908
33 days 06:29:00        5
8 days 22:44:00         1
15 days 08:27:00        1
13 days 22:37:00        1
25 days 05:45:00        1
30 days 18:53:00        1
32 days 04:51:00        1
5 days 02:42:00         1
2 days 13:48:00         1
Name: count, dtype: int64


In [ ]:
missing_rows = df[df["Global_active_power"].isna()]

print("Number of missing rows:", len(missing_rows))

display(
    missing_rows[
        ["Date", "Time", "timestamp"]
    ].head(20)
)
# I have 25,979 missing measurement rows, and they occur at specific timestamps, sometimes in consecutive blocks.
# These are not individual random missing values. There are periods where the measurement data is missing.

Number of missing rows: 25979


,Date,Time,timestamp
6839,21/12/2006,11:23:00,2006-12-21 11:23:00
6840,21/12/2006,11:24:00,2006-12-21 11:24:00
19724,30/12/2006,10:08:00,2006-12-30 10:08:00
19725,30/12/2006,10:09:00,2006-12-30 10:09:00
41832,14/1/2007,18:36:00,2007-01-14 18:36:00
61909,28/1/2007,17:13:00,2007-01-28 17:13:00
98254,22/2/2007,22:58:00,2007-02-22 22:58:00
98255,22/2/2007,22:59:00,2007-02-22 22:59:00
142588,25/3/2007,17:52:00,2007-03-25 17:52:00
190497,28/4/2007,00:21:00,2007-04-28 00:21:00


# Find the length of each missing block

In [30]:
# Create a boolean column showing whether energy data is missing
df["is_missing"] = df["Global_active_power"].isna()

# Identify changes between missing and non-missing periods
df["missing_group"] = (
    df["is_missing"] != df["is_missing"].shift()
).cumsum()

# Get the size of each missing block
missing_blocks = (
    df[df["is_missing"]]
    .groupby("missing_group")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        missing_count=("timestamp", "count")
    )
    .sort_values("missing_count", ascending=False)
)

print(missing_blocks.head(20))

                       start_time            end_time  missing_count
missing_group                                                       
138           2010-08-17 21:02:00 2010-08-22 21:27:00           7226
140           2010-09-25 03:56:00 2010-09-28 19:12:00           5237
14            2007-04-28 00:21:00 2007-04-30 14:23:00           3723
100           2009-06-13 00:30:00 2009-06-15 07:34:00           3305
118           2010-01-12 14:53:00 2010-01-14 19:01:00           3129
126           2010-03-20 03:52:00 2010-03-21 13:38:00           2027
104           2009-08-13 05:00:00 2009-08-13 19:50:00            891
32            2007-07-15 16:49:00 2007-07-15 18:11:00             83
78            2008-12-10 10:48:00 2008-12-10 11:57:00             70
34            2007-07-15 18:21:00 2007-07-15 19:07:00             47
70            2008-10-25 10:28:00 2008-10-25 11:10:00             43
84            2009-02-01 16:29:00 2009-02-01 17:06:00             38
26            2007-06-09 17:59:00 

In [ ]:
# Check the largest missing gaps
print("Total missing blocks:", len(missing_blocks)) 

Total missing blocks: 71


In [32]:
print("Largest missing block:")
display(missing_blocks.head(10))

Largest missing block:


,start_time,end_time,missing_count
missing_group,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129
126,2010-03-20 03:52:00,2010-03-21 13:38:00,2027
104,2009-08-13 05:00:00,2009-08-13 19:50:00,891
32,2007-07-15 16:49:00,2007-07-15 18:11:00,83
78,2008-12-10 10:48:00,2008-12-10 11:57:00,70


In [35]:
# Distribution of missing block sizes.
print(missing_blocks["missing_count"].describe())

print(missing_blocks.sort_values("missing_count"))

count      71.000000
mean      365.901408
std      1251.468043
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max      7226.000000
Name: missing_count, dtype: float64
                       start_time            end_time  missing_count
missing_group                                                       
18            2007-06-06 21:56:00 2007-06-06 21:56:00              1
124           2010-02-14 21:23:00 2010-02-14 21:23:00              1
116           2010-01-02 18:51:00 2010-01-02 18:51:00              1
112           2009-11-09 20:39:00 2009-11-09 20:39:00              1
80            2008-12-20 14:34:00 2008-12-20 14:34:00              1
...                           ...                 ...            ...
118           2010-01-12 14:53:00 2010-01-14 19:01:00           3129
100           2009-06-13 00:30:00 2009-06-15 07:34:00           3305
14            2007-04-28 00:21:00 2007-04-30 14:23:00           3723
140           2010-09-25 03:56:00 201

In [36]:
# 
time_diff = df["timestamp"].diff()

print(time_diff.value_counts().head(10))

timestamp
0 days 00:01:00    2075258
Name: count, dtype: int64


In [37]:
print("Expected 1-minute intervals:", (time_diff == pd.Timedelta(minutes=1)).sum())

Expected 1-minute intervals: 2075258


In [ ]:
# Missing-gap statistics
print(missing_blocks["missing_count"].describe())

count      71.000000
mean      365.901408
std      1251.468043
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max      7226.000000
Name: missing_count, dtype: float64


In [ ]:
# Time interval check
time_diff = df["timestamp"].diff()

print(time_diff.value_counts().head(10))

timestamp
0 days 00:01:00    2075258
Name: count, dtype: int64


# 7. Check invalid Negative Values and Zero Values

In [18]:
for column in numeric_columns:
    print(
        column,
        "negative values:",
        (df[column] < 0).sum()
    )

Global_active_power negative values: 0
Global_reactive_power negative values: 0
Voltage negative values: 0
Global_intensity negative values: 0
Sub_metering_1 negative values: 0
Sub_metering_2 negative values: 0
Sub_metering_3 negative values: 0


In [19]:
for column in numeric_columns:
    print(
        column,
        "zero values:",
        (df[column] == 0).sum()
    )

Global_active_power zero values: 0
Global_reactive_power zero values: 481561
Voltage zero values: 0
Global_intensity zero values: 0
Sub_metering_1 zero values: 1880175
Sub_metering_2 zero values: 1436830
Sub_metering_3 zero values: 852092


In [21]:
print("Invalid timestamps:", df["timestamp"].isna().sum())

Invalid timestamps: 0


In [22]:
timestamp_duplicates = df["timestamp"].duplicated().sum()

print("Duplicate timestamps:", timestamp_duplicates)

Duplicate timestamps: 0


In [23]:
# Check timestamp ordering
df = df.sort_values("timestamp")

print("First timestamp:", df["timestamp"].min())
print("Last timestamp:", df["timestamp"].max())

First timestamp: 2006-12-16 17:24:00
Last timestamp: 2010-11-26 21:02:00
